In [ ]:
from src.graph.process_init.tdr_parsing import TDRParsing
from src.graph.state import FormuladorCTeIAgent
import uuid

test_state = FormuladorCTeIAgent(
    tdr_document_path="G:/Mi unidad/IR Consulting/Connectnova/Proyectos CTeI/tdr convocatoria 37/terminos_de_referencia_convocatoria_37_30-04-2025.pdf"
)

config = {
    "thread_info": {
        "configurable": {
            "thread_id": str(uuid.uuid4())
        }
    }
}

doc = TDRParsing().run(test_state, config)
print(doc)


In [ ]:
from pydantic import BaseModel
from langgraph.prebuilt import create_react_agent

class WeatherResponse(BaseModel):
    conditions: str

def get_weather(city: str) -> str:  
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"


agent = create_react_agent(
    model="openai:gpt-4.1-mini",
    tools=[get_weather],
    response_format=WeatherResponse  
)

response = agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)

response["structured_response"].conditions

In [ ]:
import pkg_resources
import subprocess

packages = [dist.project_name for dist in pkg_resources.working_set]

for package in packages:
    print(f"⏫ Actualizando {package}...")
    subprocess.call(["pip", "install", "--upgrade", package])


In [ ]:
from importlib.metadata import distributions
import subprocess

packages = [dist.metadata['Name'] for dist in distributions()]

for package in packages:
    print(f"⏫ Actualizando {package}...")
    subprocess.call(["pip", "install", "--upgrade", package])

In [ ]:
"""
Crea un vector-store con FAISS, sin Chroma ni OpenTelemetry.
"""

from pathlib import Path
import pandas as pd
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

# ──────────────────────────────
# 1. Leer el Excel completo
# ──────────────────────────────
EXCEL_PATH = Path("src/databases/cat_prod_ctei.xlsx")
df = pd.read_excel(EXCEL_PATH).fillna("").astype(str)

def row_to_text(row) -> str:
    return " | ".join(f"{col}: {row[col]}" for col in df.columns)

documents = [row_to_text(r) for _, r in df.iterrows()]
metadatos = df.to_dict("records")

# ──────────────────────────────
# 2. Embeddings Hugging Face
# ──────────────────────────────
MODEL_ID = "sentence-transformers/all-MiniLM-L6-v2"
emb = HuggingFaceEmbeddings(model_name=MODEL_ID)

# ──────────────────────────────
# 3. Vector-store local con FAISS
# ──────────────────────────────
INDEX_DIR = Path("src/databases/faiss_ctei")
vectordb = FAISS.from_texts(
    texts=documents,
    embedding=emb,
    metadatas=metadatos,
)
vectordb.save_local(str(INDEX_DIR))

print(f"✅ FAISS index creado en {INDEX_DIR} con {len(documents)} filas")


In [ ]:
"""
probe_vec_retriever_full_v2.py
-----------------------------------------------------------------
Prueba profunda del tool vec_retriever_ctei (v2):

• 1er nivel:  objetivo_especifico  → lista de productos sugeridos
• 2º nivel:  cada producto        → detalle usando su Código del Producto

Requisitos
----------
pip install langchain-huggingface faiss-cpu pandas
El índice FAISS debe existir en src/databases/faiss_ctei/
El tool vec_retriever_ctei v2 debe estar importable en src.tools.catalogo_mga
"""

from pathlib import Path
import json
from collections import defaultdict

# ----------------------------------------------------------------------
# 0)  Configuración
# ----------------------------------------------------------------------
JSON_PATH      = Path("src/databases/sample_project.json")   # ⇐ ajusta la ruta
TOP_K_PRODUCTS = 5   # Nº de productos por objetivo
TOP_1_BY_CODE  = 1   # Nº de registros al buscar por código

# ----------------------------------------------------------------------
# 1)  Cargar proyecto y extraer objetivos
# ----------------------------------------------------------------------
with JSON_PATH.open(encoding="utf-8") as f:
    proyecto = json.load(f)

objetivos = proyecto.get("objetivos_especificos", [])
if not objetivos:
    raise ValueError("No se encontraron 'objetivos_especificos' en el JSON.")

# ----------------------------------------------------------------------
# 2)  Importar el tool
# ----------------------------------------------------------------------
from src.tools.catalogo_mga import vec_retriever_ctei

# ----------------------------------------------------------------------
# 3)  Construir mapping objetivo → productos (con detalle)
# ----------------------------------------------------------------------
resumen_obj_prod = defaultdict(list)

for idx, objetivo in enumerate(objetivos, 1):
    print(f"\n🎯 OBJETIVO {idx}: {objetivo}")

    # ---------- 1er nivel: productos sugeridos ----------
    hits = vec_retriever_ctei.invoke({"query": objetivo, "k": TOP_K_PRODUCTS})
    if not hits:
        print("   (sin productos relevantes)")
        continue

    for n, hit in enumerate(hits, 1):
        fields = hit["fields"]                     # ⇐ ahora es 'fields'
        codigo = fields.get("Código del Producto", "").strip()
        titulo = fields.get("Producto", "").strip() or fields.get("Descripción", "").strip()

        print(f"   {n}. {codigo} · {titulo}")

        # ---------- 2º nivel: detalle vía código ----------
        if codigo:
            det = vec_retriever_ctei.invoke({"query": codigo, "k": TOP_1_BY_CODE})[0]
            resumen_obj_prod[objetivo].append(det)

# ----------------------------------------------------------------------
# 4)  Mostrar resumen estructurado
# ----------------------------------------------------------------------
print("\n════════════════  RESUMEN OBJETIVO → PRODUCTOS  ════════════════\n")
for obj, productos in resumen_obj_prod.items():
    print(f"OBJETIVO: {obj}")
    for prod in productos:
        flds = prod["fields"]
        print(f"   • {flds.get('Código del Producto', '—')} — {flds.get('Producto', '')[:80]}")
    print("-" * 70)


In [ ]:
from src.tools.catalogo_mga import vec_retriever_ctei

query = "Red de sensores IoT para cultivos en el departamento del Atlántico"
hits = vec_retriever_ctei.invoke({"query": query, "k": 3})   # <- dict wrapper por @tool

for h in hits:
    print("\n---- REGISTRO ----")
    for k, v in h["fields"].items():
        print(f"{k}: {v}")


In [ ]:
"""
build_catalogo_ctei.py
----------------------
• Lee cat_prod_ctei.xlsx.
• Convierte cada fila en texto + metadatos.
• Calcula embeddings OpenAI text-embedding-3-small.
• Guarda índice FAISS persistente en src/databases/faiss_ctei_openai
  (no mezcla tu índice HF; así puedes tener ambos).
"""

import pandas as pd
from pathlib import Path

from langchain_openai import OpenAIEmbeddings          # v0.1.2+
from langchain.vectorstores import FAISS

# 0) Parámetros ----------------------------------------------------------------
EXCEL_PATH   = Path("src/databases/cat_prod_ctei.xlsx")
INDEX_DIR    = Path("src/databases/faiss_ctei_openai")
MODEL_NAME   = "text-embedding-3-small"                # 1536-dimensiones
COLUMNS      = None  # None → toma todas; o lista explícita

# 1) Cargar Excel ---------------------------------------------------------------
df = pd.read_excel(EXCEL_PATH).fillna("").astype(str)
if COLUMNS:                             # solo columnas deseadas
    df = df[COLUMNS]

def row_to_text(row) -> str:
    return " | ".join(f"{c}: {row[c]}" for c in df.columns)

texts      = [row_to_text(r) for _, r in df.iterrows()]
metadatas  = df.to_dict("records")

# 2) Embeddings OpenAI ----------------------------------------------------------
# Asegúrate de exportar OPENAI_API_KEY en tu terminal
emb = OpenAIEmbeddings(model=MODEL_NAME)

# 3) Construir y guardar FAISS --------------------------------------------------
vectordb = FAISS.from_texts(texts=texts, embedding=emb, metadatas=metadatas)
vectordb.save_local(str(INDEX_DIR))
print(f"✅ FAISS (OpenAI) guardado en {INDEX_DIR}  | filas: {len(texts)}")


In [1]:
import pandas as pd
from pathlib import Path

# 1. Asegúrate de que el paquete de tu proyecto esté en el PYTHONPATH
import sys, os
root = Path.cwd()
if str(root) not in sys.path:
    sys.path.append(str(root))

# 2. Importa el tool (usa la ruta de tu proyecto)
from src.tools.catalogo_mga import vec_retriever_ctei

# -------- parámetros de prueba ----------
QUERY = "sensores IoT para cultivos"
TOP_K = 5     # número de resultados para comparar
# ----------------------------------------

# Consulta con embeddings OpenAI
hits_openai = vec_retriever_ctei.invoke(
    {"query": QUERY, "k": TOP_K, "backend": "openai"}
)

# Consulta con embeddings HuggingFace
hits_hf = vec_retriever_ctei.invoke(
    {"query": QUERY, "k": TOP_K, "backend": "hf"}
)


# Helper para convertir la respuesta a DataFrame
def hits_to_df(hits):
    rows = []
    for rank, hit in enumerate(hits, 1):
        f = hit["fields"]
        rows.append(
            {
                "Rank": rank,
                "Código del Producto": f.get("Código del Producto", ""),
                "Producto / Descripción": f.get("Producto", "") or f.get("Descripción", ""),
                "Backend": hit["backend"],
            }
        )
    return pd.DataFrame(rows)


df_comparativa = pd.concat(
    [hits_to_df(hits_openai), hits_to_df(hits_hf)], ignore_index=True
)

import ace_tools as tools; tools.display_dataframe_to_user("Comparativa OpenAI vs HF", df_comparativa)


: 

In [1]:
import importlib.metadata as md, langgraph_swarm, inspect, sys, pprint
print("Version:", md.version("langgraph-swarm"))
print("File   :", langgraph_swarm.__file__)
print("First 5 suspicious lines:")
print("".join(inspect.getsource(langgraph_swarm.swarm).splitlines(True)[:130]))
print("sys.path order (top 5):"); pprint.pp(sys.path[:5])


Version: 0.0.11
File   : C:\Users\Ivan\AppData\Roaming\Python\Python313\site-packages\langgraph_swarm\__init__.py
First 5 suspicious lines:
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.pregel import Pregel
from typing_extensions import Any, Literal, Optional, Type, TypeVar, Union, get_args, get_origin

from langgraph_swarm.handoff import get_handoff_destinations


class SwarmState(MessagesState):
    """State schema for the multi-agent swarm."""

    # NOTE: this state field is optional and is not expected to be provided by the user.
    # If a user does provide it, the graph will start from the specified active agent.
    # If active agent is typed as a `str`, we turn it into enum of all active agent names.
    active_agent: Optional[str]


StateSchema = TypeVar("StateSchema", bound=SwarmState)
StateSchemaType = Type[StateSchema]


def _update_state_schema_agent_names(
    state_schema: StateSchemaType, agent_names: list[str]
) -> StateSchemaType:
    ""

In [3]:
import importlib.metadata as md, langgraph_swarm, inspect, sys, textwrap
print("▶ langgraph-swarm version:", md.version("langgraph-swarm"))
print("▶ loaded from          :", langgraph_swarm.__file__)
print("▶ critical line        :", [
        l for l in inspect.getsource(langgraph_swarm.swarm.add_active_agent_router).splitlines()
        if "builder.schemas" in l][:3])
print("▶ first five sys.path  :", *sys.path[:5], sep="\n   ")
PY


▶ langgraph-swarm version: 0.0.11
▶ loaded from          : C:\Users\Ivan\AppData\Roaming\Python\Python313\site-packages\langgraph_swarm\__init__.py


OSError: could not get source code